# turbopuffer in 10 minutes, with clinical trials

This notebook shows the four things the TrialScope app does with [turbopuffer](https://turbopuffer.com), on
**12 real studies from ClinicalTrials.gov** typed right into the notebook:

1. **Load** documents, letting turbopuffer embed the text for you
2. **Search** three ways: by keyword, by meaning, and both at once
3. **Filter** to trials a specific patient could actually join
4. **Keep each patient's notes separate**, and search a note the moment it's saved

You need a turbopuffer API key. In Colab, click the 🔑 **Secrets** icon on the left, add `TURBOPUFFER_API_KEY`,
and turn on **Notebook access**. Then run the cells top to bottom.

*Personal learning project. The patient and notes below are made up; trial records are public data and may be out of date.*

In [ ]:
%pip install -q turbopuffer==3.1.0 pandas

## Connect

In [2]:
import os
import pandas as pd
import turbopuffer

pd.set_option("display.max_colwidth", 80)

try:
    from google.colab import userdata          # running in Colab
    API_KEY = userdata.get("TURBOPUFFER_API_KEY")
except ImportError:
    API_KEY = os.environ.get("TURBOPUFFER_API_KEY") or input("turbopuffer API key: ")

tpuf = turbopuffer.Turbopuffer(api_key=API_KEY, region="aws-us-east-1")
trials_ns = tpuf.namespace("quickstart-trials")   # a namespace is like a table; it's created on first write


def show(result, *columns):
    """Turn query results into a small table."""
    rows = [{**r.model_dump(), **(r.model_extra or {})} for r in result.rows]
    df = pd.DataFrame(rows).rename(columns={"$dist": "score"})
    return df[[c for c in ("id", "score", *columns) if c in df]]

print("Connected")

Connected


## 1. The data: 12 real trials

Six breast cancer trials (most about HER2-positive disease) and six heart trials (atrial fibrillation, heart
failure, blood pressure). Each has a title, a short summary, and the fields we'll filter on later.

In [3]:
trials = [
    {"id": "NCT07407920", "title": "Ph2 Study for Optimization of Adjunct Systemic Therapy in HER2+ Patients, MolecularPCR Trial", "status": "Recruiting", "phase": "Phase 2", "min_age": 18, "max_age": 120, "sex": "All", "summary": "This phase II trial tests reduced post surgery (adjuvant) therapy for patients with early breast cancer who have confirmed that the disease has responded completely (pathologic complete response) after pre surgical treatment (neoadjuvant) therapy…"},
    {"id": "NCT04893109", "title": "ATEMPT 2.0: Adjuvant T-DM1 vs TH", "status": "Recruiting", "phase": "Phase 2", "min_age": 18, "max_age": 120, "sex": "All", "summary": "This research study is studying how well newly diagnosed breast cancer that has tested positive for a protein called HER2 responds using one of two different combination of HER2-directed therapies as a treatment after surgery."},
    {"id": "NCT06876714", "title": "ShortStop-HER2: 12 Months vs. 6 Months of HER2-targeted Medications for People With HER2+ Breast Cancer Who Had a Pathologic Complete Response After Chemotherapy Plus Trastuzumab", "status": "Recruiting", "phase": "Phase 3", "min_age": 18, "max_age": 120, "sex": "All", "summary": "This phase III trial compares 6 months of human epidermal growth factor receptor 2 (HER2)-targeted therapy to 12 months of HER2-targeted therapy for the treatment of HER2-positive (+) breast cancer in patients that had a pathologic complete response…"},
    {"id": "NCT05232916", "title": "Phase 3 Study to Evaluate the Efficacy and Safety of HER2/Neu Peptide GLSI-100 (GP2 + GM-CSF) in HER2/Neu Positive Subjects", "status": "Recruiting", "phase": "Phase 3", "min_age": 18, "max_age": 100, "sex": "All", "summary": "This is a prospective, randomized, double-blinded, placebo-controlled, multi-center, Phase 3 study of GLSI-100 immunotherapy in HLA-A*02 positive and HER2/neu positive subjects who are at high risk for disease recurrence and have completed both…"},
    {"id": "NCT05406635", "title": "Imaging Versus Cardiac Biomarker Monitored HER2 Directed Therapy in Patients With Breast Cancer", "status": "Active not recruiting", "phase": "N/A", "min_age": 18, "max_age": 90, "sex": "Female", "summary": "Due to a risk of heart failure during HER2 directed therapy in breast cancer, treatment is monitored with imaging of myocardial function, which is resource demanding for both patients and the health care system."},
    {"id": "NCT03418961", "title": "S1501 Dual Observational and Randomized Cohort Study of Patients With Metastatic HER-2+ Breast Cancer at Risk of Cardiac Toxicity", "status": "Recruiting", "phase": "Phase 3", "min_age": 18, "max_age": 120, "sex": "All", "summary": "This trial has two cohorts of patients with human epidermal growth factor receptor (HER)-2-positive breast cancer that has spread to other places in the body. All patients must be receiving trastuzumab-based treatment."},
    {"id": "NCT07783984", "title": "Thoracoscopic Versus Catheter Ablation in Persistent Atrial Fibrillation With Enlarged Left Atrium", "status": "Recruiting", "phase": "N/A", "min_age": 19, "max_age": 120, "sex": "All", "summary": "This is a prospective, single-center, open-label, randomized controlled trial comparing thoracoscopic surgical ablation with percutaneous endocardial catheter ablation as the first rhythm-control procedure in patients with persistent atrial…"},
    {"id": "NCT07783919", "title": "Single-Shot Versus Focal Dual-Energy Pulsed Field-Based Ablation for Atrial Fibrillation", "status": "Not yet recruiting", "phase": "N/A", "min_age": 18, "max_age": 120, "sex": "All", "summary": "Comparison of single-shot PFA systems with focal dual-energy (PFA/RF) in patients at high risk of atrial fibrillation recurrence."},
    {"id": "NCT07644806", "title": "Olive Oil Supplementation for Heart Failure With Reduced Ejection Fraction", "status": "Recruiting", "phase": "N/A", "min_age": 18, "max_age": 85, "sex": "All", "summary": "The OLEA-HF study aims to explore whether daily supplementation of extra-virgin olive oil (EVOO) for 12 weeks during participation in a cardiac rehabilitation program (standard of care) is feasible in patients with heart failure with reduced…"},
    {"id": "NCT07465653", "title": "A Safety and Tolerability Study of HJB647 in Heart Failure Participants With Reduced Ejection Fraction", "status": "Recruiting", "phase": "Phase 1", "min_age": 18, "max_age": 100, "sex": "All", "summary": "The purpose of this study is to evaluate the safety, tolerability, and pharmacokinetics of HJB647 at two different doses in participants with chronic stable heart failure with reduced or mildly reduced ejection fraction (HFrEF/HFmrEF)."},
    {"id": "NCT07778966", "title": "Community Health Worker Support to Lower Blood Pressure After a Pregnancy With High Blood Pressure", "status": "Recruiting", "phase": "N/A", "min_age": 16, "max_age": 56, "sex": "Female", "summary": "High blood pressure during or after pregnancy (such as preeclampsia, gestational hypertension, eclampsia, or Hemolysis, Elevated Liver enzymes, and Low Platelet (HELLP) syndrome) raises a mother's risk of heart disease later in life."},
    {"id": "NCT06034743", "title": "A Study to Investigate the Efficacy and Safety of Baxdrostat in Participants With Uncontrolled Hypertension on Two or More Medications Including Participants With Resistant Hypertension", "status": "Completed", "phase": "Phase 3", "min_age": 18, "max_age": 130, "sex": "All", "summary": "This is a Phase III, multicentre, randomised, double-blinded, placebo-controlled, parallel group study to evaluate the safety, tolerability and effect of 1 or 2 mg baxdrostat versus placebo, administered once daily (QD) orally, on the reduction of…"},
]

pd.DataFrame(trials)[["id", "title", "status", "sex", "min_age", "max_age"]]

,id,title,status,sex,min_age,max_age
0,NCT07407920,"Ph2 Study for Optimization of Adjunct Systemic Therapy in HER2+ Patients, Mo...",Recruiting,All,18,120
1,NCT04893109,ATEMPT 2.0: Adjuvant T-DM1 vs TH,Recruiting,All,18,120
2,NCT06876714,ShortStop-HER2: 12 Months vs. 6 Months of HER2-targeted Medications for Peop...,Recruiting,All,18,120
3,NCT05232916,Phase 3 Study to Evaluate the Efficacy and Safety of HER2/Neu Peptide GLSI-1...,Recruiting,All,18,100
4,NCT05406635,Imaging Versus Cardiac Biomarker Monitored HER2 Directed Therapy in Patients...,Active not recruiting,Female,18,90
5,NCT03418961,S1501 Dual Observational and Randomized Cohort Study of Patients With Metast...,Recruiting,All,18,120
6,NCT07783984,Thoracoscopic Versus Catheter Ablation in Persistent Atrial Fibrillation Wit...,Recruiting,All,19,120
7,NCT07783919,Single-Shot Versus Focal Dual-Energy Pulsed Field-Based Ablation for Atrial ...,Not yet recruiting,All,18,120
8,NCT07644806,Olive Oil Supplementation for Heart Failure With Reduced Ejection Fraction,Recruiting,All,18,85
9,NCT07465653,A Safety and Tolerability Study of HJB647 in Heart Failure Participants With...,Recruiting,All,18,100


## 2. Load them

The one idea to take away: **a single `text` field is set up two ways.** `full_text_search` indexes its words
for keyword search, and `embed` tells turbopuffer to turn it into a vector itself. We just send plain text.

The other fields are stored as-is and can be filtered on.

In [4]:
rows = [{**t, "text": f"{t['title']}. {t['summary']}"} for t in trials]

result = trials_ns.write(
    upsert_rows=rows,                      # insert, or replace rows with the same id
    distance_metric="cosine_distance",
    schema={"text": {"type": "string", "full_text_search": True, "embed": "voyage/voyage-4-lite"}},
)
print(f"Loaded {result.rows_upserted} trials")

Loaded 12 trials


## 3. Search by keyword (BM25)

Keyword search ranks by the words in the query, giving more weight to rare ones. It's exact: great for drug
names and trial ids, but it can't match a synonym. Higher score is better.

In [5]:
result = trials_ns.query(rank_by=("text", "BM25", "trastuzumab"), top_k=3, include_attributes=["title"])
show(result, "title")

,id,score,title
0,NCT03418961,1.562966,S1501 Dual Observational and Randomized Cohort Study of Patients With Metast...
1,NCT06876714,1.426120,ShortStop-HER2: 12 Months vs. 6 Months of HER2-targeted Medications for Peop...


## 4. Search by meaning (vector)

Now a question in everyday words: "chemo hurting the heart". The trials never say "chemo" or "hurting".
Vector search compares meaning instead: turbopuffer embeds the question with the same model and returns the
closest trials. Here the score is a distance, so **lower is closer**.

In [6]:
question = "chemo hurting the heart"

by_meaning = trials_ns.query(rank_by=("text", "ANN", ("Embed", question)), top_k=3, include_attributes=["title"])
by_keyword = trials_ns.query(rank_by=("text", "BM25", question), top_k=3, include_attributes=["title"])

print("By meaning:")
display(show(by_meaning, "title"))
print("By keyword:")
show(by_keyword, "title")

By meaning:


,id,score,title
0,NCT05406635,0.469473,Imaging Versus Cardiac Biomarker Monitored HER2 Directed Therapy in Patients...
1,NCT03418961,0.546968,S1501 Dual Observational and Randomized Cohort Study of Patients With Metast...
2,NCT07465653,0.590085,A Safety and Tolerability Study of HJB647 in Heart Failure Participants With...


By keyword:


,id,score,title
0,NCT07465653,1.889978,A Safety and Tolerability Study of HJB647 in Heart Failure Participants With...
1,NCT07644806,1.794172,Olive Oil Supplementation for Heart Failure With Reduced Ejection Fraction
2,NCT05406635,1.409117,Imaging Versus Cardiac Biomarker Monitored HER2 Directed Therapy in Patients...


Meaning search puts the two trials about heart problems caused by HER2 cancer treatment first. Keyword
search can only match the word "heart", so it leads with heart failure trials that have nothing to do with
cancer.

## 5. Both at once (hybrid)

The app sends both searches in **one request** and merges the two ranked lists. The merge (reciprocal rank
fusion) is simple: each trial scores `1 / (60 + rank)` for every list it appears in, so being near the top of
both lists wins.

In [7]:
question = "HER2 breast cancer treatment after surgery"

both = trials_ns.multi_query(queries=[
    {"rank_by": ("text", "ANN", ("Embed", question)), "top_k": 5},
    {"rank_by": ("text", "BM25", question), "top_k": 5},
])

scores = {}
for ranked_list in both.results:
    for rank, row in enumerate(ranked_list.rows, start=1):
        scores[row.id] = scores.get(row.id, 0) + 1 / (60 + rank)

titles = {t["id"]: t["title"] for t in trials}
pd.DataFrame([{"id": i, "score": round(s, 4), "title": titles[i]}
              for i, s in sorted(scores.items(), key=lambda x: -x[1])])

,id,score,title
0,NCT07407920,0.0325,"Ph2 Study for Optimization of Adjunct Systemic Therapy in HER2+ Patients, Mo..."
1,NCT04893109,0.0325,ATEMPT 2.0: Adjuvant T-DM1 vs TH
2,NCT06876714,0.0317,ShortStop-HER2: 12 Months vs. 6 Months of HER2-targeted Medications for Peop...
3,NCT05406635,0.0312,Imaging Versus Cardiac Biomarker Monitored HER2 Directed Therapy in Patients...
4,NCT03418961,0.0308,S1501 Dual Observational and Randomized Cohort Study of Patients With Metast...


## 6. Filter to trials a patient could join

Meet Dana, a made-up patient: 58, female, HER2-positive breast cancer. Filters are exact and run *inside* the
search, so trials she can't join never compete for a spot in the results.

In [8]:
dana = {"name": "Dana", "age": 58, "sex": "Female"}

could_join = ("And", [
    ("status", "Eq", "Recruiting"),
    ("min_age", "Lte", dana["age"]),
    ("max_age", "Gte", dana["age"]),
    ("sex", "In", [dana["sex"], "All"]),
])

result = trials_ns.query(
    rank_by=("text", "ANN", ("Embed", "HER2 breast cancer after surgery")),
    filters=could_join,
    top_k=4,
    include_attributes=["title"],
)
display(show(result, "title"))

# Which trials did the filter rule out, and why?
passed = {r.id for r in trials_ns.query(filters=could_join, top_k=20).rows}
pd.DataFrame([t for t in trials if t["id"] not in passed])[["id", "status", "min_age", "max_age", "sex", "title"]]

,id,score,title
0,NCT07407920,0.368592,"Ph2 Study for Optimization of Adjunct Systemic Therapy in HER2+ Patients, Mo..."
1,NCT04893109,0.436709,ATEMPT 2.0: Adjuvant T-DM1 vs TH
2,NCT06876714,0.457989,ShortStop-HER2: 12 Months vs. 6 Months of HER2-targeted Medications for Peop...
3,NCT03418961,0.546721,S1501 Dual Observational and Randomized Cohort Study of Patients With Metast...


,id,status,min_age,max_age,sex,title
0,NCT05406635,Active not recruiting,18,90,Female,Imaging Versus Cardiac Biomarker Monitored HER2 Directed Therapy in Patients...
1,NCT07783919,Not yet recruiting,18,120,All,Single-Shot Versus Focal Dual-Energy Pulsed Field-Based Ablation for Atrial ...
2,NCT07778966,Recruiting,16,56,Female,Community Health Worker Support to Lower Blood Pressure After a Pregnancy Wi...
3,NCT06034743,Completed,18,130,All,A Study to Investigate the Efficacy and Safety of Baxdrostat in Participants...


The second table lists every trial the filter ruled out: not recruiting, finished, or outside Dana's age.
They never compete for a spot in her results, however well their text matches.

## 7. Each patient gets their own namespace

In the app, every patient's notes live in a separate namespace. A search of Dana's notes can only ever see
Dana's notes. Here we save four short notes, then search them straight away: turbopuffer queries are
**strongly consistent**, so a note is searchable as soon as the write returns.

In [9]:
dana_notes = tpuf.namespace("quickstart-patient-dana")

notes = [
    {"id": "n1", "date": "2026-08-12", "text": "Initial consult. New HER2-positive breast cancer, stage II. Plan chemotherapy plus trastuzumab before surgery."},
    {"id": "n2", "date": "2026-08-24", "text": "Baseline echocardiogram. Heart pumping function (LVEF) 64%, normal."},
    {"id": "n3", "date": "2026-10-31", "text": "Echo before cycle 4. LVEF 58%, a small drop from baseline. No symptoms. Keep monitoring every 3 months."},
    {"id": "n4", "date": "2026-11-04", "text": "Follow-up after cycle 4. Mild fatigue. Asked about clinical trials after surgery."},
]
dana_notes.write(upsert_rows=notes, distance_metric="cosine_distance",
                 schema={"text": {"type": "string", "full_text_search": True, "embed": "voyage/voyage-4-lite"}})

result = dana_notes.query(rank_by=("text", "ANN", ("Embed", "is her heart okay?")), top_k=2, include_attributes=["date", "text"])
show(result, "date", "text")

,id,score,date,text
0,n2,0.519428,2026-08-24,"Baseline echocardiogram. Heart pumping function (LVEF) 64%, normal."
1,n3,0.594263,2026-10-31,"Echo before cycle 4. LVEF 58%, a small drop from baseline. No symptoms. Keep..."


## How this maps to the app

| In this notebook | In TrialScope |
| --- | --- |
| `text` field with `full_text_search` and `embed` | Every corpus: trials, PubMed abstracts, full-text passages, notes |
| Keyword, vector and hybrid search | The search page's mode switch (hybrid is the default) |
| RRF merge | How hybrid lists, and results from several sources, are combined |
| Filters from the patient's age and sex | "Match patient" on the trials search, and the copilot's trial matching |
| One namespace per patient | `ts-patient-0001` … `ts-patient-0100` |

The copilot uses exactly these queries as its tools: it decides what to search, with which filters, and
cites what comes back.

## Clean up

Deletes the two namespaces this notebook made.

In [10]:
for name in ["quickstart-trials", "quickstart-patient-dana"]:
    tpuf.namespace(name).delete_all()
print("Deleted")

Deleted
